# Answer Key — Flow Matching Assignment

This notebook contains reference implementations for all assignment parts and runs
every autograder test to verify correctness.

**Do not distribute to students.**

In [ ]:
import os, sys, copy
import numpy as np
import torch
import torch.nn.functional as F
import torchaudio
from torch.utils.data import DataLoader

# Make sure we can import dataset / model from the repo root
REPO = '/home/znovack/nsynth-speedrun'
if REPO not in sys.path:
    sys.path.insert(0, REPO)

from dataset import NSynthSpecDataset, wav_to_spec, spec_to_audio, FREQ_BINS, TIME_FRAMES, SR
from model  import load_flow_model, save_flow_model, FlowModelWrapper, NULL_PITCH

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

CKPT_PATH = f'{REPO}/pretrained_keyboard.pt'
model, ckpt = load_flow_model(CKPT_PATH, device=device)
print(f'Model: {ckpt["n_params"]:,} params | FREQ_BINS={FREQ_BINS} TIME_FRAMES={TIME_FRAMES} NULL_PITCH={NULL_PITCH}')
print(f'Convention: t=1 noise, t=0 data')

---
## Part 1 — Euler Sampling

In [ ]:
def euler_sample(model, x0, pitches, n_steps=50):
    """
    Euler ODE integration from noise (t=1) to data (t=0).

    x_{t-dt} = x_t - model(x_t, t, pitch) * dt
    t goes from 1 to 1/n_steps in equal decrements of dt = 1/n_steps.
    """
    dt = 1.0 / n_steps
    x  = x0.clone()
    B  = x.shape[0]
    with torch.no_grad():
        for i in range(n_steps):
            t = torch.full((B,), 1.0 - i / n_steps, device=x.device)
            v = model(x, t, pitches)
            x = x - v * dt
    return x

In [ ]:
# === Autograder Test 1 ===
torch.manual_seed(0)
x0_ag = torch.randn(4, 2, FREQ_BINS, TIME_FRAMES, device=device)
p_ag  = torch.tensor([60, 62, 64, 67], dtype=torch.long, device=device)

out1 = euler_sample(model, x0_ag.clone(), p_ag, n_steps=20)

assert out1.shape == (4, 2, FREQ_BINS, TIME_FRAMES), f'Wrong shape: {out1.shape}'
assert not torch.allclose(out1, x0_ag, atol=1e-3)
assert out1.isfinite().all()
assert out1.std() > 0.05
print(f'\u2713 euler_sample | mean={out1.mean():.4f}, std={out1.std():.4f}')

---
## Part 2a — Naive Velocity Scaling

In [ ]:
def naive_scale_sample(model, x0, pitches, n_steps=50, scale=1.0):
    """
    Euler sampling (t=1 → t=0) with velocity multiplied by `scale`.
    At scale=1.0 this is identical to euler_sample.
    """
    dt = 1.0 / n_steps
    x  = x0.clone()
    B  = x.shape[0]
    with torch.no_grad():
        for i in range(n_steps):
            t = torch.full((B,), 1.0 - i / n_steps, device=x.device)
            v = model(x, t, pitches) * scale
            x = x - v * dt
    return x

In [ ]:
# === Autograder Test 2a ===
torch.manual_seed(0)
x0_ag = torch.randn(4, 2, FREQ_BINS, TIME_FRAMES, device=device)
p_ag  = torch.tensor([60, 62, 64, 67], dtype=torch.long, device=device)

out_s1 = naive_scale_sample(model, x0_ag.clone(), p_ag, n_steps=20, scale=1.0)
out_e  = euler_sample(      model, x0_ag.clone(), p_ag, n_steps=20)
out_s2 = naive_scale_sample(model, x0_ag.clone(), p_ag, n_steps=20, scale=2.0)

assert torch.allclose(out_s1, out_e, atol=1e-5), 'scale=1.0 must match euler_sample'
assert not torch.allclose(out_s2, out_e, atol=1e-3), 'scale=2.0 should differ'
print('\u2713 naive_scale_sample | scale=1.0 matches Euler, scale=2.0 differs')

---
## Part 2b — Classifier-Free Guidance

In [ ]:
def cfg_sample(model, x0, pitches, n_steps=50, guidance_scale=1.0):
    """
    Euler sampling (t=1 → t=0) with Classifier-Free Guidance.

    At each step:
        v_cond   = model(x_t, t, pitch)
        v_uncond = model(x_t, t, NULL_PITCH)          # skip when gs=1
        v        = v_uncond + gs * (v_cond - v_uncond)
        x        = x - v * dt
    """
    dt     = 1.0 / n_steps
    x      = x0.clone()
    B      = x.shape[0]
    p_null = torch.full_like(pitches, NULL_PITCH)
    with torch.no_grad():
        for i in range(n_steps):
            t      = torch.full((B,), 1.0 - i / n_steps, device=x.device)
            v_cond = model(x, t, pitches)
            if guidance_scale == 1.0:
                v = v_cond
            else:
                v_uncond = model(x, t, p_null)
                v = v_uncond + guidance_scale * (v_cond - v_uncond)
            x = x - v * dt
    return x

In [ ]:
# === Autograder Test 2b ===
torch.manual_seed(0)
x0_ag = torch.randn(4, 2, FREQ_BINS, TIME_FRAMES, device=device)
p_ag  = torch.tensor([60, 62, 64, 67], dtype=torch.long, device=device)

out_cfg1   = cfg_sample(model, x0_ag.clone(), p_ag, n_steps=20, guidance_scale=1.0)
out_euler  = euler_sample(model, x0_ag.clone(), p_ag, n_steps=20)
out_cfg6   = cfg_sample(model, x0_ag.clone(), p_ag, n_steps=20, guidance_scale=6.0)
out_naive2 = naive_scale_sample(model, x0_ag.clone(), p_ag, n_steps=20, scale=2.0)

assert torch.allclose(out_cfg1, out_euler, atol=1e-5), 'cfg(gs=1) must equal euler'
assert not torch.allclose(out_cfg6, out_euler, atol=1e-3), 'cfg(gs=6) should differ from euler'
assert not torch.allclose(out_cfg6, out_naive2, atol=1e-3), 'CFG must differ from naive scaling'
print('\u2713 cfg_sample | gs=1.0 matches Euler, gs=6.0 differs, CFG != naive scaling')

---
## Part 3a — Heun's Method

In [ ]:
def heun_sample(model, x0, pitches, n_steps=50, guidance_scale=1.0):
    """
    Heun's method (2nd-order Runge-Kutta) from t=1 to t=0, with optional CFG.

    Per step (integrating backward):
        k1   = v(x_t,    t)
        x_hat = x_t - k1 * dt           # Euler predictor
        k2   = v(x_hat,  t - dt)
        x_{t-dt} = x_t - (k1+k2)/2 * dt
    """
    dt     = 1.0 / n_steps
    x      = x0.clone()
    B      = x.shape[0]
    p_null = torch.full_like(pitches, NULL_PITCH)

    def guided_velocity(x_, t_val):
        t_     = torch.full((B,), t_val, device=x_.device)
        v_cond = model(x_, t_, pitches)
        if guidance_scale == 1.0:
            return v_cond
        v_uncond = model(x_, t_, p_null)
        return v_uncond + guidance_scale * (v_cond - v_uncond)

    with torch.no_grad():
        for i in range(n_steps):
            t_cur  = 1.0 - i / n_steps
            t_next = 1.0 - (i + 1) / n_steps
            k1     = guided_velocity(x, t_cur)
            x_pred = x - k1 * dt
            k2     = guided_velocity(x_pred, t_next)
            x      = x - 0.5 * (k1 + k2) * dt
    return x

In [ ]:
# === Autograder Test 3a ===
torch.manual_seed(0)
x0_ag = torch.randn(4, 2, FREQ_BINS, TIME_FRAMES, device=device)
p_ag  = torch.tensor([60, 62, 64, 67], dtype=torch.long, device=device)

out_euler    = euler_sample(model, x0_ag.clone(), p_ag, n_steps=25)
out_heun     = heun_sample( model, x0_ag.clone(), p_ag, n_steps=25, guidance_scale=1.0)
out_heun_gs1 = heun_sample( model, x0_ag.clone(), p_ag, n_steps=25, guidance_scale=1.0)
out_heun_gs6 = heun_sample( model, x0_ag.clone(), p_ag, n_steps=25, guidance_scale=6.0)

assert out_heun.shape == (4, 2, FREQ_BINS, TIME_FRAMES)
assert out_heun.isfinite().all()
assert not torch.allclose(out_heun, out_euler, atol=1e-3), 'Heun must differ from Euler'
assert torch.allclose(out_heun, out_heun_gs1, atol=1e-5), 'Heun should be deterministic'
assert not torch.allclose(out_heun_gs6, out_heun, atol=1e-3), 'gs=6 should differ from gs=1'

l2 = (out_heun - out_euler).norm().item()
print(f'\u2713 heun_sample | differs from Euler (L2={l2:.3f}), CFG works, deterministic')

---
## Part 3b — RK4 (bonus)

In [ ]:
def rk4_sample(model, x0, pitches, n_steps=25, guidance_scale=1.0):
    """
    4th-order Runge-Kutta from t=1 to t=0, with optional CFG.

    Per step (integrating backward, dt > 0):
        k1 = v(x_t,              t)
        k2 = v(x_t - k1*dt/2,   t - dt/2)
        k3 = v(x_t - k2*dt/2,   t - dt/2)
        k4 = v(x_t - k3*dt,     t - dt)
        x_{t-dt} = x_t - dt/6 * (k1 + 2*k2 + 2*k3 + k4)
    """
    dt     = 1.0 / n_steps
    x      = x0.clone()
    B      = x.shape[0]
    p_null = torch.full_like(pitches, NULL_PITCH)

    def guided_velocity(x_, t_val):
        t_val = max(t_val, 0.0)          # clamp so we never query t < 0
        t_    = torch.full((B,), t_val, device=x_.device)
        v_cond = model(x_, t_, pitches)
        if guidance_scale == 1.0:
            return v_cond
        v_uncond = model(x_, t_, p_null)
        return v_uncond + guidance_scale * (v_cond - v_uncond)

    with torch.no_grad():
        for i in range(n_steps):
            t_cur = 1.0 - i / n_steps
            t_mid = t_cur - dt / 2
            t_nxt = t_cur - dt
            k1 = guided_velocity(x,                  t_cur)
            k2 = guided_velocity(x - k1 * (dt / 2),  t_mid)
            k3 = guided_velocity(x - k2 * (dt / 2),  t_mid)
            k4 = guided_velocity(x - k3 * dt,         t_nxt)
            x  = x - (dt / 6) * (k1 + 2 * k2 + 2 * k3 + k4)
    return x

In [ ]:
# === Autograder Test 3b (bonus) ===
torch.manual_seed(0)
x0_ag = torch.randn(4, 2, FREQ_BINS, TIME_FRAMES, device=device)
p_ag  = torch.tensor([60, 62, 64, 67], dtype=torch.long, device=device)

out_rk4  = rk4_sample( model, x0_ag.clone(), p_ag, n_steps=12, guidance_scale=1.0)
out_heun = heun_sample(model, x0_ag.clone(), p_ag, n_steps=25, guidance_scale=1.0)

assert out_rk4.shape == (4, 2, FREQ_BINS, TIME_FRAMES)
assert out_rk4.isfinite().all()
assert not torch.allclose(out_rk4, out_heun, atol=1e-3), 'RK4 should differ from Heun'

print(f'\u2713 rk4_sample (bonus) | differs from Heun (L2={(out_rk4-out_heun).norm():.3f})')

--- ## Part 4 — sample_timesteps + flow_loss

In [ ]:
def sample_timesteps(B: int, device, t_sample: str = 'logit_normal') -> torch.Tensor:
    if t_sample == 'logit_normal':
        return torch.sigmoid(torch.randn(B, device=device))
    else:
        return torch.rand(B, device=device)


def flow_loss(model, x_data, pitch, t, p_uncond=0.1):
    B = x_data.shape[0]

    x_noise = torch.randn_like(x_data)
    t4      = t[:, None, None, None]
    xt      = (1 - t4) * x_data + t4 * x_noise
    target  = x_noise - x_data

    pitch = pitch.clone()
    if p_uncond > 0:
        mask = torch.rand(B, device=x_data.device) < p_uncond
        pitch[mask] = NULL_PITCH

    v_pred = model(xt, t, pitch)
    return F.mse_loss(v_pred, target)

In [ ]:
# === Autograder Tests 4a + 4b ===
import copy
torch.manual_seed(0)

# ── Q4a: sample_timesteps ──────────────────────────────────────────────────
t_unif  = sample_timesteps(1000, device, 'uniform')
t_logit = sample_timesteps(1000, device, 'logit_normal')
assert t_unif.shape == (1000,) and (t_unif >= 0).all() and (t_unif <= 1).all()
assert t_logit.shape == (1000,) and (t_logit >= 0).all() and (t_logit <= 1).all()
assert (t_logit - 0.5).abs().mean() < (t_unif - 0.5).abs().mean()
print(f'\u2713 sample_timesteps | uniform={((t_unif-0.5).abs().mean()):.3f}, '
      f'logit-normal={((t_logit-0.5).abs().mean()):.3f}')

# ── Q4b: flow_loss ────────────────────────────────────────────────────────
torch.manual_seed(42)
x_data_ag = torch.randn(4, 2, FREQ_BINS, TIME_FRAMES, device=device) * 0.5
p_ag      = torch.randint(0, 128, (4,), dtype=torch.long, device=device)
t_ag      = sample_timesteps(4, device, 'logit_normal')

model_ag = copy.deepcopy(model)
opt_ag   = torch.optim.AdamW(model_ag.parameters(), lr=1e-4)

model_ag.train()
loss = flow_loss(model_ag, x_data_ag, p_ag, t_ag, p_uncond=0.0)
assert loss.shape == (), f'Must return scalar tensor, got shape {loss.shape}'
assert loss.grad_fn is not None, 'Must be differentiable'
assert 0 < loss.item() < 10, f'Loss {loss.item():.4f} out of range'

params_before = [p_.data.clone() for p_ in model_ag.parameters()]
opt_ag.zero_grad(set_to_none=True)
loss.backward()
torch.nn.utils.clip_grad_norm_(model_ag.parameters(), 1.0)
opt_ag.step()
model_ag.eval()
changed = any(not torch.allclose(pb, pa) for pb, pa in zip(params_before, model_ag.parameters()))
assert changed, 'Parameters did not change'
print(f'\u2713 flow_loss | loss={loss.item():.4f}, training step works')

---
## All tests summary

In [ ]:
# Run every autograder test in one shot and report results
results = {}

def run_test(name, fn):
    try:
        fn()
        results[name] = 'PASS'
    except Exception as e:
        results[name] = f'FAIL: {e}'

torch.manual_seed(0)
x0  = torch.randn(4, 2, FREQ_BINS, TIME_FRAMES, device=device)
p   = torch.tensor([60, 62, 64, 67], dtype=torch.long, device=device)

# --- Q1 ---
def test_q1():
    out = euler_sample(model, x0.clone(), p, n_steps=20)
    assert out.shape == (4, 2, FREQ_BINS, TIME_FRAMES)
    assert not torch.allclose(out, x0, atol=1e-3)
    assert out.isfinite().all()
    assert out.std() > 0.05
run_test('Q1: euler_sample', test_q1)

# --- Q2a ---
def test_q2a():
    s1 = naive_scale_sample(model, x0.clone(), p, n_steps=20, scale=1.0)
    e  = euler_sample(      model, x0.clone(), p, n_steps=20)
    s2 = naive_scale_sample(model, x0.clone(), p, n_steps=20, scale=2.0)
    assert torch.allclose(s1, e, atol=1e-5)
    assert not torch.allclose(s2, e, atol=1e-3)
run_test('Q2a: naive_scale_sample', test_q2a)

# --- Q2b ---
def test_q2b():
    c1 = cfg_sample(model, x0.clone(), p, n_steps=20, guidance_scale=1.0)
    e  = euler_sample(model, x0.clone(), p, n_steps=20)
    c6 = cfg_sample(model, x0.clone(), p, n_steps=20, guidance_scale=6.0)
    n2 = naive_scale_sample(model, x0.clone(), p, n_steps=20, scale=2.0)
    assert torch.allclose(c1, e, atol=1e-5)
    assert not torch.allclose(c6, e, atol=1e-3)
    assert not torch.allclose(c6, n2, atol=1e-3)
run_test('Q2b: cfg_sample', test_q2b)

# --- Q3a ---
def test_q3a():
    e   = euler_sample(model, x0.clone(), p, n_steps=25)
    h   = heun_sample( model, x0.clone(), p, n_steps=25, guidance_scale=1.0)
    h2  = heun_sample( model, x0.clone(), p, n_steps=25, guidance_scale=1.0)
    hg6 = heun_sample( model, x0.clone(), p, n_steps=25, guidance_scale=6.0)
    assert h.shape == (4, 2, FREQ_BINS, TIME_FRAMES)
    assert h.isfinite().all()
    assert not torch.allclose(h, e, atol=1e-3)
    assert torch.allclose(h, h2, atol=1e-5)
    assert not torch.allclose(hg6, h, atol=1e-3)
run_test('Q3a: heun_sample', test_q3a)

# --- Q3b ---
def test_q3b():
    r = rk4_sample( model, x0.clone(), p, n_steps=12, guidance_scale=1.0)
    h = heun_sample(model, x0.clone(), p, n_steps=25, guidance_scale=1.0)
    assert r.shape == (4, 2, FREQ_BINS, TIME_FRAMES)
    assert r.isfinite().all()
    assert not torch.allclose(r, h, atol=1e-3)
run_test('Q3b: rk4_sample', test_q3b)

# --- Q4a ---
def test_q4a():
    torch.manual_seed(0)
    tu = sample_timesteps(1000, device, 'uniform')
    tl = sample_timesteps(1000, device, 'logit_normal')
    assert tu.shape == (1000,) and (tu >= 0).all() and (tu <= 1).all()
    assert tl.shape == (1000,) and (tl >= 0).all() and (tl <= 1).all()
    assert (tl - 0.5).abs().mean() < (tu - 0.5).abs().mean()
run_test('Q4a: sample_timesteps', test_q4a)

# --- Q4b ---
def test_q4b():
    torch.manual_seed(42)
    xd = torch.randn(4, 2, FREQ_BINS, TIME_FRAMES, device=device) * 0.5
    pf = torch.randint(0, 128, (4,), dtype=torch.long, device=device)
    ta = sample_timesteps(4, device, 'logit_normal')
    mag = copy.deepcopy(model)
    oag = torch.optim.AdamW(mag.parameters(), lr=1e-4)
    pb  = [p_.data.clone() for p_ in mag.parameters()]
    mag.train()
    lv = flow_loss(mag, xd, pf, ta, p_uncond=0.0)
    assert lv.shape == () and lv.grad_fn is not None
    assert 0 < lv.item() < 10
    oag.zero_grad(set_to_none=True)
    lv.backward()
    torch.nn.utils.clip_grad_norm_(mag.parameters(), 1.0)
    oag.step()
    mag.eval()
    assert any(not torch.allclose(b, a) for b, a in zip(pb, mag.parameters()))
run_test('Q4b: flow_loss', test_q4b)

# ── Summary ───────────────────────────────────────────────────────────────────
print('\n' + '='*52)
print('  AUTOGRADER RESULTS')
print('='*52)
for name, status in results.items():
    icon = '\u2713' if status == 'PASS' else '\u2717'
    print(f'  {icon} {name}: {status}')
n_pass = sum(1 for s in results.values() if s == 'PASS')
print('='*52)
print(f'  {n_pass}/{len(results)} tests passed')